# GATraj-Refine on Google Colab
Choose **Runtime > Change runtime type > GPU**, then run cells in order.

This adds a small endpoint-and-path residual refiner to your **original trained GATraj**. Its backbone weights remain frozen. The decoder still outputs 20 trajectories. Zero-initialized corrections and bypass evaluation preserve the frozen original predictions. No improved test accuracy is claimed before full evaluation.

The workflow follows the RiskMotion/aWTA notebooks: mount Drive, configure one fold, train in chunks, automatically resume from Drive, and evaluate a predetermined final checkpoint. This notebook trains only the refiner by default and compares it against the same embedded original weights without training another baseline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone, install and check
Code runs from local Colab storage. Checkpoints are written to Drive every epoch. Focused CPU checks run before training, including frozen-backbone behavior, zero-correction identity and exact checkpoint continuation.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

DRIVE_ROOT = Path('/content/drive/MyDrive')
assert DRIVE_ROOT.is_dir(), 'Mount Drive first.'
REPO = Path('/content/GATraj-Refine')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Kedar2002/GATraj-Refine.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-c',
    "import torch, unittest; torch.set_num_threads(2); "
    "result=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.discover('tests')); "
    "raise SystemExit(0 if result.wasSuccessful() else 1)"], check=True)

## Configure one fold and choose the original checkpoint
Fold IDs: **0 ETH, 1 Hotel, 2 Zara1, 3 Zara2, 4 University**.

Set `BASE_CHECKPOINT` to the actual original GATraj checkpoint for this fold. If it is in `savedata/1/1/GATraj/`, include both `1` folders in that path. The default below is only a suggested location; it must match your Drive. Files from aWTA or RiskMotion are not original checkpoints. An original baseline arm from the aWTA repo is acceptable when its metadata confirms `awta=False` and no attention modification.

Legacy originals may lack metadata. In that case the notebook warns because it cannot verify the fold automatically; confirm that this checkpoint was trained with the selected held-out scene and the same architecture. An unreadable checkpoint must be recovered or replaced before training a refiner.

Keep `TRAIN_BACKBONE=False` to reuse your original weights. Set it to `True` only if you need this repository to train a fresh original GATraj first. Its separate schedule is 1,000 epochs; that option adds the full backbone training cost.

A refiner uses a separate **100-epoch** schedule by default. This is an unvalidated starting budget. Set `REFINER_STOP_AT_EPOCH=1` for startup while leaving `REFINER_EPOCHS=100` fixed. Temperature-based aWTA is not used.

In [ ]:
TEST_SET = 0
SEED = 1
VALIDATION_SET = -1  # -1: standard test; another scene index: separate validation pilot.
TRAIN_BACKBONE = False
BASE_CHECKPOINT = DRIVE_ROOT / 'GATraj' / 'savedata' / str(TEST_SET) / 'GATraj' / 'GATraj_1000.tar'

ENDPOINT_REFINE = True
SHAPE_REFINE = True
SHAPE_COMPONENTS = 3
MAX_ENDPOINT_CORRECTION = 1.0  # Metres per coordinate; not a tuned benchmark value.
MAX_SHAPE_CORRECTION = 0.5
ENDPOINT_LOSS_WEIGHT = 0.1
CORRECTION_PENALTY = 0.01

BACKBONE_EPOCHS = 1000
REFINER_EPOCHS = 100
REFINER_STOP_AT_EPOCH = REFINER_EPOCHS
EPOCHS_PER_RUN = 10
PRUNE_INTERMEDIATE_CHECKPOINTS = True
KEEP_EVERY = 10
OUTPUT_DIR = DRIVE_ROOT / 'GATraj-Refine' / 'runs'

## Train or automatically resume
Fresh refiner training first checks that the actual source checkpoint loads and that its initialized predictions equal bypass predictions. Base weights are embedded in every new refiner checkpoint, alongside the refiner, optimizer, scheduler, RNG and source fingerprint.

Each completed epoch is saved directly to Drive using a temporary file before replacement. After successful chunks, optional cleanup keeps every `KEEP_EVERY` epoch and the latest. Checkpoint/configuration mismatches stop instead of silently starting over. Do not run two sessions against one directory.

For a clean validation pilot, select a validation scene distinct from `TEST_SET`. The source backbone must also have excluded that scene; metadata checks reject a known mismatch or missing validation provenance. `TRAIN_BACKBONE=True` can produce that three-scene backbone. Such scores are validation results, not the standard four-scene benchmark. Keep validation selection separate from the final test experiment.

In [ ]:
assert 0 < REFINER_STOP_AT_EPOCH <= REFINER_EPOCHS
assert EPOCHS_PER_RUN > 0 and KEEP_EVERY > 0
assert ENDPOINT_REFINE or SHAPE_REFINE
assert VALIDATION_SET == -1 or (0 <= VALIDATION_SET <= 4 and VALIDATION_SET != TEST_SET)

common = [
    '--test_set', str(TEST_SET), '--seed', str(SEED), '--validation_set', str(VALIDATION_SET),
    '--x_encoder_layers', '3', '--eta_min', '1e-5', '--batch_size', '32',
    '--randomRotate', 'True', '--final_mode', '20', '--neighbor_thred', '10',
    '--using_cuda', 'True', '--gpu', '0', '--clip', '1', '--pass_time', '2',
    '--ifGaussian', 'False', '--SR', 'True', '--input_offset', 'True',
]

def stream(command):
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'Process exited with code {result}; see its actual error above.')

def resolve_run(flags, folder):
    resolver = 'from train import get_parser, experiment_name; print(experiment_name(get_parser().parse_args()))'
    name = subprocess.check_output([sys.executable, '-c', resolver, *flags], cwd=REPO, text=True).strip()
    return name, folder / str(TEST_SET) / name

def checkpoints(directory, name):
    found = {}
    for path in directory.glob(f'{name}_*.tar'):
        suffix = path.stem.rsplit('_', 1)[-1]
        if suffix.isdigit() and int(suffix) > 0:
            found[int(suffix)] = path
    return found

def validate(path, epoch):
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if int(checkpoint['epoch']) != epoch:
        raise RuntimeError(f'Epoch mismatch: {path}')
    for key in ('state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'rng_state', 'experiment_config'):
        if key not in checkpoint:
            raise RuntimeError(f'{path} lacks {key}; new refiner runs require their own complete checkpoints.')
    del checkpoint

def run_stage(flags, directory, name, stop, horizon, chunk, keep):
    available = checkpoints(directory, name)
    completed = max(available, default=0)
    if completed > horizon:
        raise RuntimeError('Checkpoint exceeds configured horizon.')
    if completed:
        validate(available[completed], completed)
    print('Checkpoint directory:', directory, 'Resume after epoch:', completed)
    while completed < stop:
        count = min(chunk, stop-completed)
        stream([sys.executable, '-u', 'train.py', *flags,
                '--epochs_per_run', str(count), '--load_model', str(completed)])
        advanced = max(checkpoints(directory, name), default=0)
        if advanced != completed + count:
            raise RuntimeError('Expected checkpoint was not saved.')
        validate(directory / f'{name}_{advanced}.tar', advanced)
        completed = advanced
        if PRUNE_INTERMEDIATE_CHECKPOINTS:
            for epoch, path in checkpoints(directory, name).items():
                if epoch < completed and epoch % keep:
                    path.unlink()
        print('Saved to Drive through epoch', completed)
    return completed

if TRAIN_BACKBONE:
    backbone_folder = OUTPUT_DIR / 'backbones'
    backbone_flags = [*common, '--stage', 'backbone', '--num_epochs', str(BACKBONE_EPOCHS),
                      '--learning_rate', '5e-4', '--save_base_dir', str(backbone_folder)]
    backbone_name, backbone_dir = resolve_run(backbone_flags, backbone_folder)
    run_stage(backbone_flags, backbone_dir, backbone_name, BACKBONE_EPOCHS, BACKBONE_EPOCHS, 100, 100)
    BASE_CHECKPOINT = backbone_dir / f'{backbone_name}_{BACKBONE_EPOCHS}.tar'

if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Set BASE_CHECKPOINT to your actual original checkpoint. Not found: {BASE_CHECKPOINT}')
# Hash the actual file without importing stale model code into the notebook process.
hash_program = 'from refinement import file_sha256; import sys; print(file_sha256(sys.argv[1]))'
base_sha256 = subprocess.check_output([sys.executable, '-c', hash_program, str(BASE_CHECKPOINT)], cwd=REPO, text=True).strip()
refiner_folder = OUTPUT_DIR / 'refiners'
refiner_flags = [
    *common, '--stage', 'refiner', '--base_checkpoint', str(BASE_CHECKPOINT), '--base_sha256', base_sha256,
    '--num_epochs', str(REFINER_EPOCHS), '--learning_rate', '1e-4',
    '--endpoint_refine', str(ENDPOINT_REFINE), '--shape_refine', str(SHAPE_REFINE),
    '--shape_components', str(SHAPE_COMPONENTS), '--max_endpoint_correction', str(MAX_ENDPOINT_CORRECTION),
    '--max_shape_correction', str(MAX_SHAPE_CORRECTION), '--endpoint_loss_weight', str(ENDPOINT_LOSS_WEIGHT),
    '--correction_penalty', str(CORRECTION_PENALTY), '--save_base_dir', str(refiner_folder),
]
run_name, run_dir = resolve_run(refiner_flags, refiner_folder)

# Real checkpoint compatibility and zero-initialization check before data processing/GPU training.
probe = r"""
import sys, torch
torch.set_num_threads(2)
from train import get_parser
from refinement import GATrajRefiner
sys.path.insert(0, 'tests')
from test_refiner import example
a=get_parser().parse_args(); a.using_cuda=False; a.load_model=0
m=GATrajRefiner(a).eval()
with torch.no_grad():
    on=m(example(),1,return_mixture=True)
    a.refiner=False
    off=m(example(),1,return_mixture=True)
    for x,y in zip(on,off): torch.testing.assert_close(x,y,rtol=0,atol=0)
print('PASS: actual original checkpoint loads; zero-initialized refinement equals bypass.')
print('Backbone parameters:',sum(p.numel() for p in m.backbone.parameters()))
print('Refiner parameters:',sum(p.numel() for p in m.refinement.parameters()))
"""
stream([sys.executable, '-c', probe, *refiner_flags])
print('Original source:', BASE_CHECKPOINT, 'SHA256:', base_sha256)
run_stage(refiner_flags, run_dir, run_name, REFINER_STOP_AT_EPOCH, REFINER_EPOCHS, EPOCHS_PER_RUN, KEEP_EVERY)

## Paired evaluation at the fixed final refiner checkpoint
This evaluates bypass and refinement using exactly the same embedded original backbone. Both use K=20 and identical test windows. Reports are stored separately. Baseline probabilities/scales are retained; their calibration after refinement is not established by ADE/FDE results.

For validation experiments, the comparison recommends bypass if either ADE or FDE worsens. For test experiments, it reports both and selects neither. Never pick the best refiner epoch using the test scene.

In [ ]:
final_path = run_dir / f'{run_name}_{REFINER_EPOCHS}.tar'
if not final_path.is_file():
    raise RuntimeError('Finish REFINER_EPOCHS before the final comparison.')
validate(final_path, REFINER_EPOCHS)
for enabled in (False, True):
    stream([sys.executable, '-u', 'evaluate.py', *refiner_flags,
            '--phase', 'test', '--load_model', str(REFINER_EPOCHS), '--refiner', str(enabled)])
comparison_path = run_dir / f'comparison_{REFINER_EPOCHS}.json'
stream([sys.executable, 'compare.py',
        str(run_dir / f'evaluation_{REFINER_EPOCHS}_baseline.json'),
        str(run_dir / f'evaluation_{REFINER_EPOCHS}_refined.json'),
        '--output', str(comparison_path)])
print('Reports and comparison saved in Drive:', run_dir)

## Ablations and reporting
Run endpoint-only (`True/False`), shape-only (`False/True`) and combined (`True/True`) on validation. The off result is always available by bypassing the refiner. Shape-only refinement leaves endpoints unchanged, so its FDE should equal the baseline by construction. Component settings create separate run directories and use the same chosen source checkpoint.

For the final fixed-setting comparison, use all five held-out folds and paired seeds where resources permit. Average folds equally within each seed. Distinguish refiner-seed variation with a fixed pretrained backbone from variation across independently trained backbones. Preserve negative results. Initialization identity and a frozen backbone do not guarantee that learned corrections improve an unseen scene.

This repository is an adaptation of existing residual/coarse-to-fine refinement ideas. It does not implement PnS, EigenTrajectory, TrajFine, RiskMotion or aWTA, and does not claim a novel or validated improvement before experiments.